# 01 · Leitura — DATASUS (SINAN Dengue + População)

**Camada 1 do ETL — Leitura.** Baixa as tabulações do TabNet e as consolida em Parquet,
**sem aplicar regras de negócio**. Nesta camada os valores continuam exatamente como a fonte
entrega (texto, com `-` para zero, linha `Total`, rótulos `"código NOME"`); a limpeza é
responsabilidade da camada de Tratamento.

| | |
|---|---|
| **Fonte** | [TabNet SINAN Dengue](https://tabnet.datasus.gov.br/cgi/deftohtm.exe?sinannet/cnv/denguebbr.def) e [Estimativas populacionais SVS/IBGE](https://tabnet.datasus.gov.br/cgi/deftohtm.exe?ibge/cnv/popsvs2024br.def) |
| **Entrada** | consultas HTTP ao TabNet → `data/raw/datasus/tabnet/**.csv` |
| **Saída** | `data/01_leitura/datasus/*.parquet` |
| **Próxima etapa** | `02_tratamento/01_tratamento_datasus.ipynb` |

**Por que só 29 consultas?** O formulário tem 47 linhas × 34 colunas × 13 anos × 57 filtros.
Todas as opções ficam registradas no *catálogo* (dicionário de dados), mas só baixamos as
tabulações que respondem à pergunta do projeto: **o clima antecipa a explosão de casos de dengue?**

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))

import download_datasus  # noqa: E402
from config import LEITURA, RAW_DATASUS, garantir_pastas  # noqa: E402

garantir_pastas()
SAIDA = LEITURA / "datasus"
SAIDA.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 20)

## 1. Download (idempotente)

`coletar()` relê o formulário (atualizando o catálogo) e executa só as consultas que ainda não
foram baixadas com sucesso — cada uma fica registrada no `manifesto.csv`.
Para forçar uma nova coleta completa, use `REFAZER = True` (leva ~15 min).

In [2]:
REFAZER = False
manifesto = download_datasus.coletar(workers=2, refazer=REFAZER)

Catálogo: 47 linhas, 34 colunas, 1 conteúdo(s), 13 períodos, 57 filtros -> data/raw/datasus/catalogo
29 consultas nos conjuntos [1, 2, 3, 4, 5]; 0 pendentes
Concluído em 0.0 min. 29 ok, 0 com erro. Manifesto: data/raw/datasus/tabnet/manifesto.csv


In [3]:
man = pd.read_csv(RAW_DATASUS / "tabnet" / "manifesto.csv", sep=";")
man["arquivo"] = man["arquivo"].str.replace("data/raw/datasus/tabnet/", "", regex=False)
grupos = {1: "1 · Município × semana", 2: "2 · Contexto anual", 3: "3 · Perfil e gravidade",
          4: "4 · Desigualdade", 5: "5 · População"}
man["conjunto"] = man["conjunto"].map(grupos)
man[["conjunto", "arquivo", "n_linhas", "n_colunas", "total_casos", "segundos", "coletado_em", "status"]]

,conjunto,arquivo,n_linhas,n_colunas,total_casos,segundos,coletado_em,status
0,1 · Município × semana,01_municipio_semana/2014.csv,5599,55,543917,1.5,2026-11-15T22:12:27,ok
1,1 · Município × semana,01_municipio_semana/2015.csv,5599,54,1623172,2.6,2026-11-15T22:12:25,ok
2,1 · Município × semana,01_municipio_semana/2016.csv,5599,54,1450080,2.3,2026-11-15T22:12:23,ok
3,1 · Município × semana,01_municipio_semana/2017.csv,5599,54,239395,39.4,2026-11-15T22:16:21,ok
4,1 · Município × semana,01_municipio_semana/2018.csv,5599,54,262611,0.9,2026-11-15T22:09:58,ok
5,1 · Município × semana,01_municipio_semana/2019.csv,5599,54,1546252,2.4,2026-11-15T22:09:57,ok
6,1 · Município × semana,01_municipio_semana/2020.csv,5599,55,975841,2.8,2026-11-15T22:09:55,ok
7,1 · Município × semana,01_municipio_semana/2021.csv,5599,54,540049,1.2,2026-11-15T22:12:21,ok
8,1 · Município × semana,01_municipio_semana/2022.csv,5599,54,1405094,24.6,2026-11-15T22:09:52,ok
9,1 · Município × semana,01_municipio_semana/2023.csv,5599,54,1645350,24.9,2026-11-15T22:09:27,ok


## 2. Catálogo do formulário (dicionário de dados)

Registro de **todas** as configurações possíveis do TabNet, inclusive as não utilizadas.

In [4]:
cat = RAW_DATASUS / "catalogo"
dimensoes = pd.read_csv(cat / "dimensoes.csv", sep=";")
print(f"{len(dimensoes)} variáveis no formulário: "
      f"{dimensoes.como_linha.sum()} disponíveis como linha, {dimensoes.como_coluna.sum()} como coluna, "
      f"{dimensoes.como_filtro.sum()} como filtro")
dimensoes.head(12)

61 variáveis no formulário: 47 disponíveis como linha, 33 como coluna, 57 como filtro


,variavel,rotulo,como_linha,como_coluna,como_filtro,n_categorias_filtro
0,Ano_1º_Sintoma(s),Ano 1º Sintoma(s),1,1,1,54.0
1,Mês_1º_Sintoma(s),Mês 1º Sintoma(s),1,1,1,13.0
2,Semana_epidem._1º_Sintomas(s),Semana epidem. 1º Sintomas(s),1,1,1,54.0
3,Ano_notificação,Ano notificação,1,1,1,54.0
4,Mês_notificação,Mês notificação,1,1,1,13.0
5,Semana_epidem._notificação,Semana epidem. notificação,1,1,1,54.0
6,Ano_epidem._notificação,Ano epidem. notificação,1,1,1,54.0
7,Ano_epidem._1º_Sintomas(s),Ano epidem. 1º Sintomas(s),1,1,1,54.0
8,Região_de_notificação,Região de notificação,1,1,1,6.0
9,Região/UF_de_notificação,Região/UF de notificação,1,0,0,NaN


In [5]:
atualizacao = json.loads((cat / "atualizacao_por_ano.json").read_text(encoding="utf-8"))
pd.DataFrame(atualizacao.items(), columns=["ano", "dados atualizados em"])

,ano,dados atualizados em
0,2014,13/07/2015
1,2015,27/09/2016
2,2016,25/07/2017
3,2017,18/07/2018
4,2018,01/10/2019
5,2019,"10/07/2020 às 12:00 horas, incluídos os dados ..."
6,2020,"28/07/2021 às 03:00 horas, incluídos os dados ..."
7,2021,"12/07/2022 às 03:00 horas, incluídos os dados ..."
8,2022,"18/07/2023 às 03:00 horas, incluídos os dados ..."
9,2023,"04/06/2024 às 11:30 horas, incluídos os dados ..."


In [6]:
print((cat / "notas_fonte.txt").read_text(encoding="utf-8")[:2500])

Fonte: Ministério da Saúde/SVSA - Sistema de Informação de Agravos de Notificação - Sinan Net Notas: Para os casos prováveis foram incluídas todas notificações, exceto casos descartados. As bases de dados de dengue aqui disponíveis compõem um banco único a partir de 2014, podendo haver pequenas divergências com os dados disponibilizados pelo CGARB (Coordenação Geral de Vigilância de Arboviroses - CGARB) em sua série histórica, que para tal, realiza análise ano a ano, com a base congelada para cada ano analisado. Para tabular os casos graves (classificação final igual a dengue com complicações, febre hemorrágica da dengue, síndrome do choque da dengue, dengue com sinais de alarme e dengue grave) é necessário considerar também o critério de confirmação (laboratorial e clínico-epidemiológico). A partir de 2020 o estado do Espírito Santo passou a utilizar o sistema e-SUS Vigilância em Saúde. Portanto, para os casos de Arboviroses urbanas do Espírito Santo foram considerados apenas os dados

## 3. Leitura dos CSVs para Parquet

Todos os campos são lidos como **texto** (`dtype=str`) para preservar o conteúdo original
— inclusive o `-` que o TabNet usa para zero. Acrescentamos apenas colunas de linhagem
(`arquivo_origem`, `coletado_em`).

In [7]:
TABNET = RAW_DATASUS / "tabnet"
coleta = dict(zip(man["arquivo"], man["coletado_em"]))


def ler_tabnet(caminho: Path) -> pd.DataFrame:
    df = pd.read_csv(caminho, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
    rel = str(caminho.relative_to(TABNET))
    return df.assign(arquivo_origem=rel, coletado_em=coleta.get(rel, ""))

### 3.1 Núcleo: município de residência × semana epidemiológica (13 arquivos anuais → 1 tabela)

In [8]:
partes = []
for csv in sorted((TABNET / "01_municipio_semana").glob("*.csv")):
    partes.append(ler_tabnet(csv).assign(ano_epi=csv.stem))
municipio_semana = pd.concat(partes, ignore_index=True)
# O TabNet omite colunas de semanas sem nenhum caso no país (ex.: 'Semana 53' em anos de 52 semanas,
# semanas futuras de 2026). Após o concat essas colunas ficam vazias (NaN) no ano em que não existiam —
# a regra para interpretá-las (zero x não observado) é aplicada no Tratamento.
ausentes = municipio_semana.groupby("ano_epi").apply(lambda d: [c for c in d.columns if d[c].isna().all()])
print(ausentes[ausentes.str.len() > 0].to_string())
municipio_semana.to_parquet(SAIDA / "dengue_municipio_semana.parquet", index=False)
print(municipio_semana.shape)
municipio_semana.iloc[:3, [0, 1, 2, 3, -5, -4, -3, -2, -1]]

ano_epi
2015                                          [Semana 53]
2016                                          [Semana 53]
2017                                          [Semana 53]
2018                                          [Semana 53]
2019                                          [Semana 53]
2021                                          [Semana 53]
2022                                          [Semana 53]
2023                                          [Semana 53]
2024                                          [Semana 53]
2026    [Semana 45, Semana 46, Semana 47, Semana 48, S...
(72787, 58)


,Município de residência,Semana 01,Semana 02,Semana 03,Semana 53,Total,arquivo_origem,coletado_em,ano_epi
0,MUNICIPIO IGNORADO - RO,-,-,-,-,-,01_municipio_semana/2014.csv,2026-11-15T22:12:27,2014
1,110001 ALTA FLORESTA D'OESTE,-,-,1,-,29,01_municipio_semana/2014.csv,2026-11-15T22:12:27,2014
2,110002 ARIQUEMES,3,-,3,-,62,01_municipio_semana/2014.csv,2026-11-15T22:12:27,2014


### 3.2 Demais tabulações (um Parquet por CSV)

In [9]:
saidas = {
    "02_contexto_anual": "contexto",
    "03_perfil_gravidade": "perfil",
    "04_desigualdade": "desigualdade",
    "05_populacao": "populacao",
}
registros = [("dengue_municipio_semana.parquet", *municipio_semana.shape)]
for pasta, prefixo in saidas.items():
    for csv in sorted((TABNET / pasta).glob("*.csv")):
        df = ler_tabnet(csv)
        nome = f"{prefixo}__{csv.stem}.parquet"
        df.to_parquet(SAIDA / nome, index=False)
        registros.append((nome, *df.shape))
pd.DataFrame(registros, columns=["arquivo", "linhas", "colunas"])

,arquivo,linhas,colunas
0,dengue_municipio_semana.parquet,72787,58
1,contexto__capital_residencia_x_ano.parquet,28,17
2,contexto__municipio_residencia_x_ano.parquet,5572,17
3,contexto__regiao_residencia_x_ano.parquet,7,17
4,contexto__semana_epidemiologica_x_ano.parquet,54,17
5,contexto__uf_residencia_x_ano.parquet,29,17
6,perfil__ano_x_class_final.parquet,14,9
7,perfil__ano_x_evolucao.parquet,14,9
8,perfil__ano_x_faixa_etaria.parquet,14,16
9,perfil__ano_x_ocorreu_hospitalizacao.parquet,14,7


## 4. Checagens da leitura

A última linha de cada tabulação é o `Total` calculado pelo próprio TabNet. Conferimos que ele
bate com o total registrado no manifesto no momento da coleta — garantia de que nada se perdeu
entre o download e o Parquet.

In [10]:
def total_da_tabela(df: pd.DataFrame) -> str:
    """Último valor da linha 'Total' (coluna Total, ou o último ano quando a tabela não tem coluna Total)."""
    dados = df.drop(columns=["arquivo_origem", "coletado_em"])
    linha = dados[dados.iloc[:, 0] == "Total"]
    return linha.iloc[0, -1] if len(linha) else ""


checagem = []
for csv_rel, total_manifesto in zip(man["arquivo"], man["total_casos"]):
    df = ler_tabnet(TABNET / csv_rel)
    checagem.append((csv_rel, str(total_manifesto), total_da_tabela(df)))
checagem = pd.DataFrame(checagem, columns=["arquivo", "total_manifesto", "total_lido"])
checagem["confere"] = checagem.total_manifesto == checagem.total_lido
assert checagem.confere.all(), checagem[~checagem.confere]
print(f"{checagem.confere.sum()}/{len(checagem)} tabulações conferem com o manifesto.")

29/29 tabulações conferem com o manifesto.


**Resumo da camada de leitura (DATASUS):** 29 tabulações lidas e gravadas em
`data/01_leitura/datasus/`, com o conteúdo original preservado e totais conferidos.
Próximo passo: **`02_tratamento/01_tratamento_datasus.ipynb`**.